# La grande Consultation — analyse des réponses

This notebook runs the analyses of `src/grande_consultation.py` step by step, with tables and charts.
**Running**: "Run all", or Shift+Enter cell by cell.
The complete results can also be exported to Excel (last section).

Code and comments are in English; table headers, labels and chart text stay in French because they are the report content.

⚠ The data contain answers from minors: do not version or share them (the `data/` folder is in `.gitignore`).

## 0. Paramètres

In [ ]:
from pathlib import Path
import sys

DATA_PATH = "data/gc_reponses.xlsx"  # ← questionnaire export (path from the project root)
OUTPUT_DIR = "resultats/gc"  # ← Excel / CSV export folder

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
DATA_PATH, OUTPUT_DIR = ROOT / DATA_PATH, ROOT / OUTPUT_DIR
print("Racine du projet :", ROOT)

In [ ]:
import warnings
import matplotlib.pyplot as plt
import pandas as pd

import grande_consultation as g

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30, "display.max_colwidth", 70, "display.width", 160)
pd.options.display.float_format = "{:.3f}".format
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Chargement et structure

In [ ]:
df = g.load(DATA_PATH)  # raw export, repeated Submission IDs removed
qs = g.discover_questions(df)  # {question number: description}
g.check_structure(qs, df)  # stops with an explanation if key questions are missing
g.explode_text_answers(df, qs)  # one boolean column per option for the multi-choice questions
print(f"{len(df)} répondant·es, {len(qs)} questions")
if len(df) < 100:
    print("⚠ échantillon réduit : résultats indicatifs seulement")

# Overview of the questions found
pd.DataFrame(
    [
        {"Q": n, "type": q["type"], "intitulé": q["text"], "nb options": len(q.get("options", {}))}
        for n, q in sorted(qs.items())
    ]
)

In [ ]:
d, unmapped = g.recode(df, qs)  # clean table: scores, age bracket, profile variables
for n, vals in unmapped.items():
    print(f"⚠ Q{n} : libellés non reconnus pour le score : {vals}")
d.head()

## 2. Qualité des données
Share of missing values per column (the conditional questions are **normally** empty for part of the respondents).

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0.05].head(20).to_frame("part de NaN")

## 3. Profil de l'échantillon
Counts with a 95 % confidence interval (Wilson).

In [ ]:
freq = g.frequency_tables(df, d, qs)


def freq_of(n):
    """Frequency table of question n, largest answer first."""
    t = freq[freq["question"] == n].sort_values("%", ascending=False)
    return t[["réponse", "effectif", "base", "%", "IC95 bas", "IC95 haut"]].reset_index(drop=True)


def plot_bars(t, title, col="%", n_max=15):
    """Quick horizontal bar chart with error bars (first column = labels)."""
    t = t.head(n_max).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 1))
    err = None
    if "IC95 bas" in t:
        err = [(t[col] - t["IC95 bas"]).clip(lower=0), (t["IC95 haut"] - t[col]).clip(lower=0)]
    ax.barh(
        t.iloc[:, 0].astype(str).str.slice(0, 60),
        t[col] * 100,
        xerr=None if err is None else [e * 100 for e in err],
        color="#4C78A8",
        capsize=2,
    )
    ax.set_xlabel("% des répondant·es")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


for n in g.PROFILE:
    if n in qs:
        print(f"\nQ{n} — {qs[n]['text']}")
        display(freq_of(n))

## 4. Tris à plat — questions à choix unique
Pick a question in `N` to display its table and chart.

In [ ]:
single = {n: q["text"] for n, q in sorted(qs.items()) if q["type"] == "single"}
pd.Series(single, name="intitulé").to_frame()

In [ ]:
N = 3  # ← question number
t = freq_of(N)
display(t)
plot_bars(t, f"Q{N} — {qs[N]['text']}")

## 5. Questions à choix multiples
Percentage **of respondents** (the sum can exceed 100 %). Q5 is a ranking: see below.

In [ ]:
multi = g.multiple_choice_tables(df, qs)
multi_questions = sorted(int(n) for n in multi["question"].unique())
print("Questions à choix multiples :", multi_questions)

In [ ]:
M = multi_questions[0]  # ← question number
t = multi[multi["question"] == M].sort_values("effectif", ascending=False)
display(t[["option", "effectif", "base", "% des répondants", "IC95 bas", "IC95 haut"]].reset_index(drop=True))
plot_bars(
    t.rename(columns={"option": "réponse"})[["réponse", "% des répondants", "IC95 bas", "IC95 haut"]],
    f"Q{M} — {qs[M]['text']}",
    col="% des répondants",
)

In [ ]:
q5 = g.ranking_table(qs)
display(q5)

In [ ]:
near_urgent = g.near_vs_urgent_causes(df, qs)
near_urgent

## 6. Croisements : questions × profil
Distribution of each single-choice question within each profile group (`% du groupe`).
Groups with fewer than `MIN_N` = 20 respondents are flagged `effectif trop faible`: do not draw conclusions from them.

In [ ]:
detail_x = g.simple_crossings(d, qs)
print("Questions croisées :", detail_x["question"].nunique(), "· variables de profil :", sorted(detail_x["variable_profil"].unique()))

In [ ]:
Q_X, VAR = "Q3", "genre"  # ← question ("Q" + number) and profile variable to detail
detail = (
    detail_x[(detail_x["question"] == Q_X) & (detail_x["variable_profil"] == VAR)] if not detail_x.empty else detail_x
)
detail.head(40)

In [ ]:
cm = g.multiple_crossings(df, d, qs)
Q_M, VAR_M = multi_questions[0], "genre"  # ← multiple-choice question and profile variable
t = cm[(cm["question"] == Q_M) & (cm["variable_profil"] == VAR_M)]
table = t.pivot_table(index="option", columns="groupe", values="% du groupe")
table.style.format("{:.0%}").background_gradient(cmap="Blues", axis=None)

## 7. Intention de vote (Q23)
Base: people old enough to vote.

In [ ]:
vote = g.voting_analysis(d)
vote

## 8. Profils types (classification)
Hierarchical clustering (Ward) on the ordinal scores; the number of profiles is chosen by the silhouette.
A silhouette < 0.25 means the profiles are weakly marked: present them with caution.

In [ ]:
profiles_table, quality, _ = g.typical_profiles(d)
display(quality)
display(profiles_table)

## 9. Départements et régions

In [ ]:
depts = g.departments(d)
depts

In [ ]:
# Same counts per région (sum of the départements); the maps are in gc_graphiques.ipynb, section 2b
g.regions(d)

## 10. Question ouverte (Q13)

In [ ]:
answers, words = g.free_text_words(df, qs)
display(answers.head(15))
words.head(25)

In [ ]:
if not words.empty:
    top = words.head(20).iloc[::-1]
    top.plot.barh(
        x="mot", y="occurrences", legend=False, figsize=(7, 5), color="#4C78A8", title="Mots les plus fréquents (Q13)"
    )
    plt.show()

## 11. Export Excel et CSV
Writes the same files as `python src/grande_consultation.py`: `resultats.xlsx` and `donnees_propres.csv`.

In [ ]:
sheets = {
    "Tris à plat": freq,
    "Choix multiples": multi,
    "Q5 classement": q5,
    "Causes proche vs urgente": near_urgent,
    "Croisements": detail_x,
    "Multiples x profil": cm,
    "Intention de vote (Q23)": vote,
    "Profils types": profiles_table,
    "Profils - qualité": quality,
    "Départements": depts,
    "Régions": g.regions(d),
    "Q13 réponses": answers,
    "Q13 mots": words,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_DIR / "resultats.xlsx") as xl:
    for name, t in sheets.items():
        if t is not None and not t.empty:
            t.to_excel(xl, sheet_name=name[:31], index=False)  # Excel limits sheet names to 31 characters
d.to_csv(OUTPUT_DIR / "donnees_propres.csv", index=False)
print("Écrit dans", OUTPUT_DIR)